# DistilBERT fine-tune

Since this is standalone in kaggle, some functions are replicated

In [ ]:
!pip install -q "transformers>=4.45,<5" "datasets>=3.0" emoji onnx onnxruntime accelerate

In [ ]:
import numpy as np
from pathlib import Path
import onnxruntime as ort
from collections import Counter
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers import AutoTokenizer
from datasets import load_dataset, Dataset
import json, shutil, re, html, unicodedata, emoji, torch
from sklearn.utils.class_weight import compute_class_weight
from onnxruntime.quantization import QuantType, quantize_dynamic

In [ ]:
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only"
print("device: ", device)

# Hyperparameter Configs

In [ ]:
BASE_MODEL = "distilbert-base-uncased"
DATASET = "cardiffnlp/tweet_eval"
DATASET_CONFIG = "sentiment"
LABELS = {0: "negative", 1: "neutral", 2: "positive"}

MAX_LENGTH = 96
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
SEED = 42
FP16 = torch.cuda.is_available()
METRIC_FOR_BEST = "eval_macro_f1"

OUTPUT_DIR = "/kaggle/working"
CONFIG = {
    "base_model": BASE_MODEL,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "seed": SEED,
    "fp16": FP16,
    "metric_for_best_model": METRIC_FOR_BEST,
}
CONFIG

# Cleaning the Data

In [ ]:
ESCAPE_RE = re.compile(r"\\u([0-9a-fA-F]{4})")
URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w{1,15}\b")
WS_RE = re.compile(r"\s+")


def _escaped_char(match):
    return chr(int(match.group(1), 16))


def expand_escapes(text):
    """Expand escaped unicode and HTML entities until the text stops changing."""
    while True:
        expanded = html.unescape(ESCAPE_RE.sub(_escaped_char, text))
        if expanded == text:
            return text
        text = expanded


def normalize(text):
    """Normalize text while preserving sentiment-bearing case and punctuation."""
    normalized = unicodedata.normalize("NFKC", expand_escapes(text))
    normalized = URL_RE.sub("[url]", normalized)
    normalized = MENTION_RE.sub("[user]", normalized)
    normalized = emoji.demojize(normalized, delimiters=("[", "]"))
    return WS_RE.sub(" ", normalized).strip()


checks = {
    r"today\u002c not perfect": "today, not perfect",
    "Fish &amp; chips": "Fish & chips",
    "Hi @alice https://example.com": "Hi [user] [url]",
    "  \uff37\uff2f\uff37  ": "WOW",
}
for raw, expected in checks.items():
    got = normalize(raw)
    assert got == expected, f"{raw!r} -> {got!r}, expected {expected!r}"
print("cleaner self-checks passed")
for raw in list(checks)[:3]:
    print(f"{raw!r} -> {normalize(raw)!r}")

## Load and clean

In [ ]:
raw = load_dataset(DATASET, DATASET_CONFIG)
data = {}
for split in ("train", "validation", "test"):
    rows = raw[split]
    data[split] = {
        "text": [normalize(str(t)) for t in rows["text"]],
        "label": [int(v) for v in rows["label"]],
    }
    print(f"{split:11s} {len(data[split]['label']):6,} rows")

print()
for text in data["train"]["text"][:3]:
    print(" ", text[:90])

## 4. Class weights

In [ ]:
train_labels = data["train"]["label"]
counts = Counter(train_labels)
class_weights = compute_class_weight(
    "balanced", classes=np.array(sorted(LABELS)), y=np.array(train_labels)
)
for label_id in sorted(LABELS):
    share = counts[label_id] / len(train_labels) * 100
    name = LABELS[label_id]
    weight = class_weights[label_id]
    print(f"{name:9s} {counts[label_id]:6,}  {share:5.2f}%   weight {weight:.4f}")

CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32)

## 5. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def encode(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = {}
for split, rows in data.items():
    dataset = Dataset.from_dict({"text": rows["text"], "labels": rows["label"]})
    tokenized[split] = dataset.map(encode, batched=True, remove_columns=["text"])

lengths = [len(ids) for ids in tokenized["train"]["input_ids"]]
percentiles = [np.percentile(lengths, q) for q in (50, 95, 99)]
print(
    f"wordpiece p50 {percentiles[0]:.0f}  p95 {percentiles[1]:.0f}  "
    f"p99 {percentiles[2]:.0f}  max {max(lengths)}"
)
truncated = sum(n >= MAX_LENGTH for n in lengths) / len(lengths)
print(f"truncated at {MAX_LENGTH}: {truncated:.3%}")

## 6. Weighted trainer

In [ ]:
set_seed(SEED)

class WeightedTrainer(Trainer):
    """Applies inverse-frequency class weights inside the loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = CLASS_WEIGHTS.to(outputs.logits.device)
        loss = F.cross_entropy(outputs.logits, labels, weight=weights)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(prediction.label_ids, predicted),
        "macro_f1": f1_score(prediction.label_ids, predicted, average="macro"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=LABELS,
    label2id={name: i for i, name in LABELS.items()},
)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 7. Train

In [ ]:
arguments = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    seed=SEED,
    fp16=FP16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST,
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to=[],
)

trainer = WeightedTrainer(
    model=model,
    args=arguments,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result.metrics)

## Reference scores

In [ ]:
fp32_scores = {}
for split in ("validation", "test"):
    metrics = trainer.evaluate(tokenized[split], metric_key_prefix=split)
    fp32_scores[split] = {
        "accuracy": float(metrics[f"{split}_accuracy"]),
        "macro_f1": float(metrics[f"{split}_macro_f1"]),
    }
    print(
        f"{split:11s} accuracy {fp32_scores[split]['accuracy']:.4f}   "
        f"macro-F1 {fp32_scores[split]['macro_f1']:.4f}"
    )

## Export to ONNX, then quantize to int8

In [ ]:
export_dir = Path(OUTPUT_DIR) / "artifacts"
export_dir.mkdir(parents=True, exist_ok=True)
fp32_path = export_dir / "model_fp32.onnx"
int8_path = export_dir / "model_int8.onnx"

model.eval().cpu()
sample = tokenizer(
    "a sample tweet for tracing",
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
)

torch.onnx.export(
    model,
    (sample["input_ids"], sample["attention_mask"]),
    str(fp32_path),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "logits": {0: "batch"},
    },
    opset_version=17,
    do_constant_folding=True,
)
print(f"fp32 {fp32_path.stat().st_size / 1_000_000:.1f} MB")

quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8)
print(f"int8 {int8_path.stat().st_size / 1_000_000:.1f} MB")

## Check the export against PyTorch

In [ ]:
session = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
probe = [normalize(t) for t in raw["test"]["text"][:256]]
encoded = tokenizer(
    probe, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="np"
)

onnx_logits = session.run(
    ["logits"],
    {
        "input_ids": encoded["input_ids"].astype("int64"),
        "attention_mask": encoded["attention_mask"].astype("int64"),
    },
)[0]
torch_inputs = tokenizer(
    probe, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt"
)
with torch.no_grad():
    torch_logits = model(**torch_inputs).logits.numpy()

agreement = (onnx_logits.argmax(-1) == torch_logits.argmax(-1)).mean()
print(f"int8 vs fp32 label agreement on {len(probe)} rows: {agreement:.2%}")
print("quantization moves a few borderline rows; macro-F1 is measured locally")

## Package for download

In [ ]:
tokenizer.save_pretrained(export_dir / "tokenizer")
fp32_path.unlink()

summary = {
    "model_name": "distilbert",
    "base_model": BASE_MODEL,
    "dataset": f"{DATASET}/{DATASET_CONFIG}",
    "config": CONFIG,
    "fp32_scores": fp32_scores,
    "train_metrics": {
        key: float(value)
        for key, value in train_result.metrics.items()
        if isinstance(value, int | float)
    },
    "log_history": trainer.state.log_history,
    "int8_vs_fp32_label_agreement": float(agreement),
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "onnxruntime": onnxruntime.__version__,
        "emoji": emoji.__version__,
    },
}
(export_dir / "training_summary.json").write_text(json.dumps(summary, indent=2))

archive = shutil.make_archive(f"{OUTPUT_DIR}/distilbert_artifacts", "zip", export_dir)
print(f"{archive}  {Path(archive).stat().st_size / 1_000_000:.1f} MB")
for path in sorted(export_dir.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(export_dir)}  {path.stat().st_size / 1_000_000:.2f} MB")

Unzip into `model/artifacts/distilbert/` locally and run `make eval`.